## **Setup**

In [9]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [10]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

## **Imports**

In [11]:
from Challenge.paths import load_xgboost_cv_folds, XGBOOST_MODELS
from Challenge.utils import load_models

## **Load Data**

In [12]:
URM_inner, URM_outer, folds_inner = load_xgboost_cv_folds()

## **Train Models**

In [13]:
from Recommenders.NonPersonalizedRecommender import TopPop
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_WARP_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_BPR_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_SVDpp_Cython

from Recommenders.SLIM.Cython.SLIM_BPR_Cython import SLIM_BPR_Cython
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender
from Recommenders.MatrixFactorization.NMFRecommender import NMFRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask

### **Candidate Generators**

In [ ]:
candidate_mapping = {
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'UserKNN_asymmetric': UserKNNCFRecommender,
    'RP3beta': RP3betaRecommender,
    'IALS': IALSRecommender,
    'TopPop': TopPop
}

candidate_cutoff = {
    'SLIMElasticNet': 80,
    'ItemKNN_tversky': 50,
    'UserKNN_asymmetric': 50,
    'RP3beta': 50,
    'IALS': 50,
    'TopPop': 50
}

In [ ]:
folder = os.path.join(XGBOOST_MODELS, "candidates")
inner_folder = os.path.join(folder, "inner")
outer_folder = os.path.join(folder, "outer")
folds_folder = os.path.join(XGBOOST_MODELS, "folds") 

#### **Inner URM**

In [ ]:
# Load models (lazy loading)
models = load_models(
    URM_train=URM_inner,
    mapping=candidate_mapping,
    model_folder=inner_folder
)

# Force loading of models
# Will train them if needed
[name for name, _ in models]

#### **Outer URM**

In [ ]:
# Load models (lazy loading)
models = load_models(
    URM_train=URM_outer,
    mapping=candidate_mapping,
    model_folder=outer_folder
)

# Force loading of models
# Will train them if needed
[name for name, _ in models]

#### **Folds**

In [ ]:
for i, (URM_train, URM_val) in enumerate(folds_inner):
    print(f"FOLD {i+1}/{len(folds_inner)}")
    
    # Load models (lazy loading)
    models = load_models(
        URM_train=URM_train,
        mapping=candidate_mapping,
        model_folder=os.path.join(folds_folder, f"fold_{i}")
    )
    
    # Force loading of models
    # Will train them if needed
    [name for name, _ in models]

### **Features Generators**

In [14]:
models_mapping = {
    'TopPop': TopPop,
    'ItemKNN_cosine': ItemKNNCFRecommender,
    'ItemKNN_jaccard': ItemKNNCFRecommender,
    'ItemKNN_asymmetric': ItemKNNCFRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'ItemKNN_dice': ItemKNNCFRecommender,
    'UserKNN_cosine': UserKNNCFRecommender,
    'UserKNN_jaccard': UserKNNCFRecommender,
    'UserKNN_asymmetric': UserKNNCFRecommender,
    'UserKNN_tversky': UserKNNCFRecommender,
    'UserKNN_dice': UserKNNCFRecommender,
    'EASE_R': EASE_R_Recommender,
    'P3alpha': P3alphaRecommender,
    'RP3beta': RP3betaRecommender,
    # 'IALS': IALSRecommender,
    'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython,
    'MatrixFactorization_BPR': MatrixFactorization_BPR_Cython,
    'MatrixFactorization_SVDpp': MatrixFactorization_SVDpp_Cython,
    
    'SLIM_BPR': SLIM_BPR_Cython,
    'NMF': NMFRecommender,
    'MultVAE': MultVAERecommender_PyTorch_OptimizerMask,
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender
}

In [15]:
folder = os.path.join(XGBOOST_MODELS, "train_features")
inner_folder = os.path.join(folder, "inner")
outer_folder = os.path.join(folder, "outer")
folds_folder = os.path.join(folder, "folds") 

#### **Inner URM**

In [8]:
# Load models (lazy loading)
models = load_models(
    URM_train=URM_inner,
    mapping=models_mapping,
    model_folder=inner_folder
)

# Force loading of models
# Will train them if needed
[name for name, _ in models]

Model not found.
  Training model: TopPop
    Could not load parameters for TopPop: [Errno 2] No such file or directory: '/home/luigi/RecSys/performance_logs/TopPop.json'
    Using default parameters
TopPopRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerTopPop'
TopPopRecommender: Saving complete
Model not found.
  Training model: ItemKNN_cosine
Similarity column 6969 (100.0%), 4657.82 column/sec. Elapsed time 1.50 sec
ItemKNNCFRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerItemKNN_cosine'
ItemKNNCFRecommender: Saving complete
Model not found.
  Training model: ItemKNN_jaccard
Similarity column 6969 (100.0%), 4691.90 column/sec. Elapsed time 1.49 sec
ItemKNNCFRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerItemKNN_jaccard'
ItemKNNCFRecommender: Saving complete
Model not found.
  Training model: ItemKNN_asymmetric
Similarity column 6969 (100.0%), 4

/home/luigi/RecSys/Recommenders/Neural/MultVAE_PyTorch_Recommender.py:330: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  user_batch_tensor = torch.sparse_csr_tensor(user_batch_tensor.indptr,


MultVAERecommender_PyTorch: Epoch 1 of 10. Elapsed time 0.80 sec
MultVAERecommender_PyTorch: Epoch 2 of 10. Elapsed time 1.30 sec
MultVAERecommender_PyTorch: Epoch 3 of 10. Elapsed time 1.81 sec
MultVAERecommender_PyTorch: Epoch 4 of 10. Elapsed time 2.31 sec
MultVAERecommender_PyTorch: Epoch 5 of 10. Elapsed time 2.82 sec
MultVAERecommender_PyTorch: Epoch 6 of 10. Elapsed time 3.33 sec
MultVAERecommender_PyTorch: Epoch 7 of 10. Elapsed time 3.84 sec
MultVAERecommender_PyTorch: Epoch 8 of 10. Elapsed time 4.35 sec
MultVAERecommender_PyTorch: Epoch 9 of 10. Elapsed time 4.86 sec
MultVAERecommender_PyTorch: Epoch 10 of 10. Elapsed time 5.37 sec
MultVAERecommender_PyTorch: Terminating at epoch 10. Elapsed time 5.38 sec
Model not found.
  Training model: SLIMElasticNet


100%|█████████▉| 6968/6969 [01:50<00:00, 62.94it/s]


SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerSLIMElasticNet'
SLIMElasticNetRecommender: Saving complete
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerTopPop'
TopPopRecommender: Loading complete
Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerItemKNN_asymmetric'
ItemKNNCFRecommender: Loading complete
Unloading ItemK

['TopPop',
 'ItemKNN_cosine',
 'ItemKNN_jaccard',
 'ItemKNN_asymmetric',
 'ItemKNN_tversky',
 'ItemKNN_dice',
 'UserKNN_cosine',
 'UserKNN_jaccard',
 'UserKNN_asymmetric',
 'UserKNN_tversky',
 'UserKNN_dice',
 'EASE_R',
 'P3alpha',
 'RP3beta',
 'MatrixFactorization_WARP',
 'MatrixFactorization_BPR',
 'MatrixFactorization_SVDpp',
 'SLIM_BPR',
 'NMF',
 'MultVAE',
 'SLIMElasticNet']

#### **Inner + Outer URM**

In [ ]:
# Load models (lazy loading)
models = load_models(
    URM_train=URM_inner+URM_outer,
    mapping=models_mapping,
    model_folder=outer_folder
)

# Force loading of models
# Will train them if needed
[name for name, _ in models]

Model not found.
  Training model: TopPop
    Could not load parameters for TopPop: [Errno 2] No such file or directory: '/home/luigi/RecSys/performance_logs/TopPop.json'
    Using default parameters
TopPopRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerTopPop'
TopPopRecommender: Saving complete
Model not found.
  Training model: ItemKNN_cosine
Similarity column 6969 (100.0%), 3453.61 column/sec. Elapsed time 2.02 sec
ItemKNNCFRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerItemKNN_cosine'
ItemKNNCFRecommender: Saving complete
Model not found.
  Training model: ItemKNN_jaccard
Similarity column 6969 (100.0%), 3497.75 column/sec. Elapsed time 1.99 sec
ItemKNNCFRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerItemKNN_jaccard'
ItemKNNCFRecommender: Saving complete
Model not found.
  Training model: ItemKNN_asymmetric
Similarity column 6969 (100.0%), 3

/home/luigi/RecSys/Recommenders/Neural/MultVAE_PyTorch_Recommender.py:330: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  user_batch_tensor = torch.sparse_csr_tensor(user_batch_tensor.indptr,


MultVAERecommender_PyTorch: Epoch 1 of 10. Elapsed time 0.70 sec
MultVAERecommender_PyTorch: Epoch 2 of 10. Elapsed time 1.22 sec
MultVAERecommender_PyTorch: Epoch 3 of 10. Elapsed time 1.74 sec
MultVAERecommender_PyTorch: Epoch 4 of 10. Elapsed time 2.27 sec
MultVAERecommender_PyTorch: Epoch 5 of 10. Elapsed time 2.79 sec
MultVAERecommender_PyTorch: Epoch 6 of 10. Elapsed time 3.31 sec
MultVAERecommender_PyTorch: Epoch 7 of 10. Elapsed time 3.83 sec
MultVAERecommender_PyTorch: Epoch 8 of 10. Elapsed time 4.35 sec
MultVAERecommender_PyTorch: Epoch 9 of 10. Elapsed time 4.87 sec
MultVAERecommender_PyTorch: Epoch 10 of 10. Elapsed time 5.41 sec
MultVAERecommender_PyTorch: Terminating at epoch 10. Elapsed time 5.41 sec
Model not found.
  Training model: SLIMElasticNet


100%|█████████▉| 6968/6969 [02:48<00:00, 41.24it/s]


SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerSLIMElasticNet'
SLIMElasticNetRecommender: Saving complete
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerTopPop'
TopPopRecommender: Loading complete
Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerItemKNN_asymmetric'
ItemKNNCFRecommender: Loading complete
Unloading ItemK

['TopPop',
 'ItemKNN_cosine',
 'ItemKNN_jaccard',
 'ItemKNN_asymmetric',
 'ItemKNN_tversky',
 'ItemKNN_dice',
 'UserKNN_cosine',
 'UserKNN_jaccard',
 'UserKNN_asymmetric',
 'UserKNN_tversky',
 'UserKNN_dice',
 'EASE_R',
 'P3alpha',
 'RP3beta',
 'MatrixFactorization_WARP',
 'MatrixFactorization_BPR',
 'MatrixFactorization_SVDpp',
 'SLIM_BPR',
 'NMF',
 'MultVAE',
 'SLIMElasticNet']

#### **Folds**

In [16]:
for i, (URM_train, URM_val) in enumerate(folds_inner):
    print(f"FOLD {i+1}/{len(folds_inner)}")
    
    # Load models (lazy loading)
    models = load_models(
        URM_train=URM_train,
        mapping=models_mapping,
        model_folder=os.path.join(folds_folder, f"fold_{i}")
    )
    
    # Force loading of models
    # Will train them if needed
    [name for name, _ in models]

FOLD 1/5
Model found: TopPop
Model found: ItemKNN_cosine
Model found: ItemKNN_jaccard
Model found: ItemKNN_asymmetric
Model found: ItemKNN_tversky
Model found: ItemKNN_dice
Model found: UserKNN_cosine
Model found: UserKNN_jaccard
Model found: UserKNN_asymmetric
Model found: UserKNN_tversky
Model found: UserKNN_dice
Model found: EASE_R
Model found: P3alpha
Model found: RP3beta
Model found: MatrixFactorization_WARP
Model found: MatrixFactorization_BPR
Model found: MatrixFactorization_SVDpp
Model found: SLIM_BPR
Model found: NMF
Model found: MultVAE
Model found: SLIMElasticNet
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_0TopPop'
TopPopRecommender: Loading complete
Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_0ItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_cosine...
Loa

100%|█████████▉| 6968/6969 [01:23<00:00, 83.00it/s] 


SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_1SLIMElasticNet'
SLIMElasticNetRecommender: Saving complete
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_1TopPop'
TopPopRecommender: Loading complete
Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_1ItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_1ItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_1ItemKNN_asymmetric'
ItemKNNCFRecommende

100%|█████████▉| 6968/6969 [01:21<00:00, 85.24it/s] 


SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_2SLIMElasticNet'
SLIMElasticNetRecommender: Saving complete
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_2TopPop'
TopPopRecommender: Loading complete
Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_2ItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_2ItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_2ItemKNN_asymmetric'
ItemKNNCFRecommende

100%|█████████▉| 6968/6969 [01:18<00:00, 88.94it/s] 


SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_3SLIMElasticNet'
SLIMElasticNetRecommender: Saving complete
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_3TopPop'
TopPopRecommender: Loading complete
Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_3ItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_3ItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_3ItemKNN_asymmetric'
ItemKNNCFRecommende

100%|█████████▉| 6968/6969 [01:18<00:00, 88.67it/s] 


SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_4SLIMElasticNet'
SLIMElasticNetRecommender: Saving complete
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_4TopPop'
TopPopRecommender: Loading complete
Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_4ItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_4ItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_4ItemKNN_asymmetric'
ItemKNNCFRecommende